In [ ]:
import pandas as pd
from datetime import datetime

In [ ]:
file_path = "data/Kabupaten Kota (0).xlsx"  # Ganti dengan path file Excel yang sebenarnya
komponens = {'adhb': (0, 17), 'adhk': (24, 41), 'distribusi': (48, 65), 'implisit': (72, 89), 'growth_yony': (96, 113), 'growth_qtoq': (120, 137), 'growth_ctoc': (144, 161), 'growth_implisit_yony': (168, 185), 'growth_implisit_qtoq': (192, 209)}

Data provinsi

In [ ]:
sheet_name = 'Prov'
df_prov = {}

In [ ]:
df = pd.read_excel(file_path, sheet_name, skiprows=4)
df.fillna(0, inplace=True)

start_col = df.columns.get_loc(datetime.now().year-5)
end_col = df.columns.get_loc(datetime.now().year-1)+1
for komponen, (start_row, end_row) in komponens.items():
    df_prov[f'df_prov_{komponen}'] = df.iloc[start_row:end_row, start_col:end_col].apply(pd.to_numeric, errors='coerce')

Data kabupaten kota

In [ ]:
df_kabkot = {}
for kabkot in range(1, 39):
    sheet_name = f'0{kabkot}' if kabkot < 10 else f'{kabkot}'
    df = pd.read_excel(file_path, sheet_name, skiprows=4)
    df.fillna(0, inplace=True)

    start_col = df.columns.get_loc(datetime.now().year-5)
    end_col = df.columns.get_loc(datetime.now().year-1)+1
    for komponen, (start_row, end_row) in komponens.items():
        if df.iloc[start_row:end_row, start_col:end_col].eq(0).all().all():
            continue
        df_kabkot[f'df_kabkot_{sheet_name}_{komponen}'] = df.iloc[start_row:end_row, start_col:end_col].apply(pd.to_numeric, errors='coerce')

Data rekap/total kabupaten kota per komponen

1. ADHB
2. ADHK

In [ ]:
df_kabkot_rekap = {}
df_kabkot_rekap_adhb = []
df_kabkot_rekap_adhk = []
for komponen, df in df_kabkot.items():
    if 'adhb' in komponen:
        df_kabkot_rekap_adhb.append(df)
    elif 'adhk' in komponen:
        df_kabkot_rekap_adhk.append(df)

df_kabkot_rekap['df_kabkot_rekap_adhb'] = sum(df_kabkot_rekap_adhb)
df_kabkot_rekap['df_kabkot_rekap_adhk'] = sum(df_kabkot_rekap_adhk)

print(df_kabkot_rekap['df_kabkot_rekap_adhk'])

3. Distribusi

In [ ]:
# Gantilah 0 menjadi 1 untuk menghindari ZeroDivisionError
denominator = df_kabkot_rekap['df_kabkot_rekap_adhb'].iloc[-1].replace(0, 1)

# Lakukan pembagian
df_kabkot_rekap_distribusi = df_kabkot_rekap['df_kabkot_rekap_adhb'].div(denominator) * 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_distribusi'] = df_kabkot_rekap_distribusi

print(df_kabkot_rekap['df_kabkot_rekap_distribusi'])

4. Implisit

In [ ]:
# Gantilah 0 menjadi 1 untuk menghindari ZeroDivisionError
denominator = df_kabkot_rekap['df_kabkot_rekap_adhk'].replace(0, 1)

# Melakukan reindexing denominator sesuai dengan index df_kabkot_rekap['df_kabkot_rekap_adhb']
denominator.index = df_kabkot_rekap['df_kabkot_rekap_adhb'].index

# Lakukan pembagian
df_kabkot_rekap_implisit = df_kabkot_rekap['df_kabkot_rekap_adhb'].div(denominator) * 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_implisit'] = df_kabkot_rekap_implisit

print(df_kabkot_rekap['df_kabkot_rekap_implisit'].info())

5. Growth Y-on-Y

In [ ]:
# Assign adhk_current_year dan adhk_prev_year
adhk_current_year = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 5:]
adhk_prev_year = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, :df_kabkot_rekap['df_kabkot_rekap_adhk'].shape[1] - 5]

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_year.columns = adhk_current_year.columns

# Lakukan pembagian
df_kabkot_rekap_growth_yony = adhk_current_year.div(adhk_prev_year) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_yony'] = df_kabkot_rekap_growth_yony

print(df_kabkot_rekap['df_kabkot_rekap_growth_yony'])

6. Growth Q-to-Q

In [ ]:
# Assign adhk_current_quarter
adhk_current_quarter = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 6:]
for i in range(1, 4):
    del adhk_current_quarter[datetime.now().year - i]  
# print(adhk_current_quarter.info())

# Assign adhk_prev_quarter
adhk_prev_quarter = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 4:]
for i in range(1, 5):
    del adhk_prev_quarter[datetime.now().year - i] 
del adhk_prev_quarter[adhk_prev_quarter.columns[-1]]
# print(adhk_prev_quarter.info())

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_quarter.columns = adhk_current_quarter.columns

# Lakukan pembagian
df_kabkot_rekap_growth_qtoq = adhk_current_quarter.div(adhk_prev_quarter) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_qtoq'] = df_kabkot_rekap_growth_qtoq

print(df_kabkot_rekap['df_kabkot_rekap_growth_qtoq'])

7. Growth C-to-C

In [ ]:
# Assign adhk_current_cumulative
adhk_current_cumulative = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 6:].astype(float)

# Identifikasi kolom dengan pola 'Q1.x, Q2.x, Q3.x, Q4.x'
pattern_cols = [col for col in df_kabkot_rekap['df_kabkot_rekap_adhk'].columns if isinstance(col, str) and '.' in col]

# Dapatkan semua suffix unik (1, 2, 3, ...)
suffixes = set(col.split('.')[1] for col in pattern_cols)

for index, row in adhk_current_cumulative.iterrows():
    for suffix in suffixes:
        q1_col = f'Q1.{suffix}'
        q2_col = f'Q2.{suffix}'
        q3_col = f'Q3.{suffix}'
        q4_col = f'Q4.{suffix}'
        
        # Pastikan kolom ada di DataFrame sebelum memproses
        if q1_col in adhk_current_cumulative.columns and q2_col in adhk_current_cumulative.columns and q3_col in adhk_current_cumulative.columns and q4_col in adhk_current_cumulative.columns:
            q1 = row[q1_col]
            q2 = row[q2_col]
            q3 = row[q3_col]
            q4 = row[q4_col]

            # Terapkan aturan perubahan
            adhk_current_cumulative.at[index, q1_col] = q1
            adhk_current_cumulative.at[index, q2_col] = q1 + q2
            adhk_current_cumulative.at[index, q3_col] = q1 + q2 + q3
            adhk_current_cumulative.at[index, q4_col] = q1 + q2 + q3 + q4

# print(adhk_current_cumulative.info())

# Assign adhk_prev_cumulative
adhk_prev_cumulative = df_kabkot_rekap['df_kabkot_rekap_adhk'].iloc[:, 1:df.shape[1] - 5].astype(float)

# Identifikasi kolom dengan pola 'Q1.x, Q2.x, Q3.x, Q4.x'
pattern_cols = [col for col in df_kabkot_rekap['df_kabkot_rekap_adhk'].columns if isinstance(col, str) and '.' in col]

# Dapatkan semua suffix unik (1, 2, 3, ...)
suffixes = set(col.split('.')[1] for col in pattern_cols)

for index, row in adhk_prev_cumulative.iterrows():
    for suffix in suffixes:
        q1_col = f'Q1.{suffix}'
        q2_col = f'Q2.{suffix}'
        q3_col = f'Q3.{suffix}'
        q4_col = f'Q4.{suffix}'
        
        # Pastikan kolom ada di DataFrame sebelum memproses
        if q1_col in adhk_prev_cumulative.columns and q2_col in adhk_prev_cumulative.columns and q3_col in adhk_prev_cumulative.columns and q4_col in adhk_prev_cumulative.columns:
            q1 = row[q1_col]
            q2 = row[q2_col]
            q3 = row[q3_col]
            q4 = row[q4_col]

            # Terapkan aturan perubahan
            adhk_prev_cumulative.at[index, q1_col] = q1
            adhk_prev_cumulative.at[index, q2_col] = q1 + q2
            adhk_prev_cumulative.at[index, q3_col] = q1 + q2 + q3
            adhk_prev_cumulative.at[index, q4_col] = q1 + q2 + q3 + q4

# print(adhk_prev_cumulative.info())

# Menyamakan kolom adhk_current_year dan adhk_prev_year
adhk_prev_cumulative.columns = adhk_current_cumulative.columns

# Lakukan pembagian
df_kabkot_rekap_growth_ctoc = adhk_current_cumulative.div(adhk_prev_cumulative) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_ctoc'] = df_kabkot_rekap_growth_ctoc

print(df_kabkot_rekap['df_kabkot_rekap_growth_ctoc'])

8. Growth implisit Y-on-Y

In [ ]:
# Assign implisit_current_year dan implisit_prev_year
implisit_current_year = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 5:]
implisit_prev_year = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, :df_kabkot_rekap['df_kabkot_rekap_implisit'].shape[1] - 5]

# Menyamakan kolom implisit_current_year dan implisit_prev_year
implisit_prev_year.columns = implisit_current_year.columns

# Lakukan pembagian
df_kabkot_rekap_growth_implisit_yony = implisit_current_year.div(implisit_prev_year) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_implisit_yony'] = df_kabkot_rekap_growth_implisit_yony

print(df_kabkot_rekap['df_kabkot_rekap_growth_implisit_yony'])

9. Growth implisit Q-to-Q

In [ ]:
# Assign implisit_current_quarter
implisit_current_quarter = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 6:]
for i in range(1, 4):
    del implisit_current_quarter[datetime.now().year - i]  
# print(implisit_current_quarter.info())

# Assign implisit_prev_quarter
implisit_prev_quarter = df_kabkot_rekap['df_kabkot_rekap_implisit'].iloc[:, 4:]
for i in range(1, 5):
    del implisit_prev_quarter[datetime.now().year - i] 
del implisit_prev_quarter[implisit_prev_quarter.columns[-1]]
# print(implisit_prev_quarter.info())

# Menyamakan kolom implisit_current_year dan implisit_prev_year
implisit_prev_quarter.columns = implisit_current_quarter.columns

# Lakukan pembagian
df_kabkot_rekap_growth_implisit_qtoq = implisit_current_quarter.div(implisit_prev_quarter) * 100 - 100

# Masukkan data dalam dictionary df_kabkot_rekap
df_kabkot_rekap['df_kabkot_rekap_growth_implisit_qtoq'] = df_kabkot_rekap_growth_implisit_qtoq

print(df_kabkot_rekap['df_kabkot_rekap_growth_implisit_qtoq'])

In [ ]:
# # Pemeriksaan index dan columns
# print(adhk_current_cumulative.index.equals(adhk_prev_cumulative.index))  # Harus True
# print(adhk_current_cumulative.columns.equals(adhk_prev_cumulative.columns))  # Harus True